# PET-SOL MD and harmonic heat-capacity analysis

This notebook visualizes only the MOF-5 + 100 CH4 calculations made with `pet_sol-s-best_nostress`. It deliberately ignores the older PET-MAD directories. It compares MD temperature and energy traces, harmonic heat-capacity curves, vibrational frequencies, and a sampled trajectory.

In [ ]:
import json
from pathlib import Path
import re
import sys

from IPython.display import Markdown, display
import matplotlib.pyplot as plt
import numpy as np

plt.style.use('seaborn-v0_8-whitegrid')

# PET-SOL selection and user settings
RUN_PATTERN = 'mof5-100ch4-*K-pet-sol-s-best-nostress-test'
HEAT_CAPACITY_PATTERN = 'heat-capacity-pet-sol-s-best-nostress-frame-*.npz'
EQUILIBRATION_FRACTION = 0.5
SELECTED_TEMPERATURE_K = 300
TRAJECTORY_STRIDE = 100
ZERO_FREQUENCY_TOLERANCE_CM1 = 1e-6


def find_project_dir(start=Path.cwd()):
    for candidate in [start, *start.parents]:
        if (candidate / 'output').is_dir() and (candidate / 'run.py').is_file():
            return candidate.resolve()
    raise FileNotFoundError('Could not locate the mof-heat-capacity project directory')


PROJECT_DIR = find_project_dir()
OUTPUT_DIR = PROJECT_DIR / 'output'
if not 0.0 <= EQUILIBRATION_FRACTION < 1.0:
    raise ValueError('EQUILIBRATION_FRACTION must be in [0, 1)')
if TRAJECTORY_STRIDE < 1:
    raise ValueError('TRAJECTORY_STRIDE must be positive')
print(f'Project: {PROJECT_DIR}')
print(f'Python kernel: {sys.executable}')
print('Potential: pet_sol-s-best_nostress')
print(f'Run pattern: {RUN_PATTERN}')

## Discover PET-SOL runs

Only directories matching `RUN_PATTERN` and heat-capacity archives matching `HEAT_CAPACITY_PATTERN` are loaded.

In [ ]:
TEMPERATURE_PATTERN = re.compile(r'(?<![0-9.])(\d+(?:\.\d+)?)K', re.IGNORECASE)


def infer_temperature(name):
    match = TEMPERATURE_PATTERN.search(name)
    return float(match.group(1)) if match else np.nan


def read_md_log(path):
    values = np.atleast_2d(np.loadtxt(path, skiprows=1, dtype=float))
    if values.shape[1] != 5 or not np.all(np.isfinite(values)):
        raise ValueError(f'Invalid MD log: {path}')
    return {
        'time_ps': values[:, 0], 'total_energy_eV': values[:, 1],
        'potential_energy_eV': values[:, 2], 'kinetic_energy_eV': values[:, 3],
        'temperature_K': values[:, 4],
    }


def scalar_text(value):
    return str(np.asarray(value).reshape(-1)[0])


def read_heat_capacity(path):
    required = {'temperatures_K', 'cv_J_per_gK', 'frame_indices', 'frequencies_cm1'}
    with np.load(path, allow_pickle=False) as archive:
        missing = required.difference(archive.files)
        if missing:
            raise ValueError(f'{path} is missing {sorted(missing)}')
        result = {name: np.asarray(archive[name]) for name in required}
        result['run_name'] = scalar_text(archive['run_name']) if 'run_name' in archive else path.parent.name
        result['metadata'] = json.loads(scalar_text(archive['metadata'])) if 'metadata' in archive else {}
    result['path'] = path
    result['temperatures_K'] = result['temperatures_K'].astype(float).reshape(-1)
    result['cv_J_per_gK'] = np.atleast_2d(result['cv_J_per_gK'].astype(float))
    result['frame_indices'] = result['frame_indices'].astype(int).reshape(-1)
    result['frequencies_cm1'] = np.atleast_2d(result['frequencies_cm1'].astype(float))
    if result['cv_J_per_gK'].shape != (len(result['frame_indices']), len(result['temperatures_K'])):
        raise ValueError(f'Inconsistent heat-capacity dimensions in {path}')
    return result


runs = []
for directory in sorted(OUTPUT_DIR.glob(RUN_PATTERN)):
    log_paths = sorted(directory.glob('*.log'))
    trajectory_paths = sorted(directory.glob('*.traj'))
    heat_paths = sorted(directory.glob(HEAT_CAPACITY_PATTERN))
    runs.append({
        'name': directory.name, 'directory': directory,
        'temperature_K': infer_temperature(directory.name),
        'md': [read_md_log(path) | {'path': path} for path in log_paths],
        'trajectories': trajectory_paths,
        'heat_capacity': [read_heat_capacity(path) for path in heat_paths],
    })
runs.sort(key=lambda run: run['temperature_K'])
if not runs:
    raise FileNotFoundError(f'No PET-SOL runs matched {OUTPUT_DIR / RUN_PATTERN}')

lines = [
    '| run | MD T [K] | logs | trajectories | PET-SOL C_V files |',
    '|---|---:|---:|---:|---:|',
]
for run in runs:
    lines.append(
        f"| `{run['name']}` | {run['temperature_K']:g} | {len(run['md'])} | "
        f"{len(run['trajectories'])} | {len(run['heat_capacity'])} |"
    )
display(Markdown('\n'.join(lines)))

## Molecular-dynamics diagnostics

The summary uses the final `1 - EQUILIBRATION_FRACTION` fraction of every PET-SOL trajectory log.

In [ ]:
md_records = [(run, md) for run in runs for md in run['md']]
if not md_records:
    print('No PET-SOL MD logs found.')
else:
    colors = plt.cm.viridis(np.linspace(0.08, 0.92, len(md_records)))
    fig, axes = plt.subplots(1, 3, figsize=(15, 4.2), layout='constrained')
    summary = []
    for (run, md), color in zip(md_records, colors, strict=True):
        label = f"{run['temperature_K']:g} K"
        time = md['time_ps']
        axes[0].plot(time, md['temperature_K'], color=color, label=label)
        axes[1].plot(time, md['total_energy_eV'] - md['total_energy_eV'][0], color=color)
        axes[2].plot(time, md['potential_energy_eV'] - md['potential_energy_eV'][0], color=color)
        start = min(int(len(time) * EQUILIBRATION_FRACTION), len(time) - 1)
        selection = slice(start, None)
        drift = np.polyfit(time[selection], md['total_energy_eV'][selection], 1)[0]
        summary.append((run['temperature_K'], np.mean(md['temperature_K'][selection]), np.std(md['temperature_K'][selection]), drift))
    axes[0].set(xlabel='Time [ps]', ylabel='Temperature [K]', title='Temperature traces')
    axes[1].set(xlabel='Time [ps]', ylabel=r'$E_{tot}(t)-E_{tot}(0)$ [eV]', title='Total-energy change')
    axes[2].set(xlabel='Time [ps]', ylabel=r'$E_{pot}(t)-E_{pot}(0)$ [eV]', title='Potential-energy change')
    axes[0].legend(title='PET-SOL run')
    plt.show()

    lines = ['| target T [K] | production mean T [K] | T std [K] | E drift [eV/ps] |', '|---:|---:|---:|---:|']
    lines.extend(f'| {target:g} | {mean:.2f} | {std:.2f} | {drift:.4g} |' for target, mean, std, drift in summary)
    display(Markdown('\n'.join(lines)))

## PET-SOL harmonic heat capacity and frequencies

Every curve is loaded from a model-labeled PET-SOL archive. The marker plot evaluates each curve at the temperature of the MD trajectory from which its structure was sampled.

In [ ]:
heat_records = [(run, result) for run in runs for result in run['heat_capacity']]
if not heat_records:
    print(f'No PET-SOL heat-capacity files matched {HEAT_CAPACITY_PATTERN}.')
else:
    colors = plt.cm.plasma(np.linspace(0.08, 0.9, len(heat_records)))
    fig, axes = plt.subplots(1, 2, figsize=(12, 4.5), layout='constrained')
    table_rows = []
    for (run, result), color in zip(heat_records, colors, strict=True):
        temperatures = result['temperatures_K']
        curves = result['cv_J_per_gK']
        label = f"{run['temperature_K']:g} K MD"
        for curve in curves:
            axes[0].plot(temperatures, curve, color=color, alpha=0.25)
        mean_curve = curves.mean(axis=0)
        axes[0].plot(temperatures, mean_curve, color=color, linewidth=2, label=label)
        matched = np.array([np.interp(run['temperature_K'], temperatures, curve) for curve in curves])
        axes[1].errorbar(run['temperature_K'], matched.mean(), yerr=matched.std(ddof=1) if len(matched) > 1 else None, fmt='o', color=color, capsize=3)
        table_rows.append((run, result, matched))
    axes[0].set(xlabel='Analysis temperature [K]', ylabel=r'$C_V$ [J g$^{-1}$ K$^{-1}$]', title='PET-SOL harmonic heat capacity')
    axes[1].set(xlabel='MD sampling temperature [K]', ylabel=r'$C_V(T_{MD})$ [J g$^{-1}$ K$^{-1}$]', title=r'$C_V$ at sampling temperature')
    axes[0].legend()
    plt.show()

    lines = ['| MD T [K] | archive | frame | C_V(T_MD) [J/gK] |', '|---:|---|---:|---:|']
    for run, result, matched in table_rows:
        frames = ', '.join(map(str, result['frame_indices']))
        lines.append(f"| {run['temperature_K']:g} | `{result['path'].name}` | {frames} | {matched.mean():.6f} |")
    display(Markdown('\n'.join(lines)))

    fig, ax = plt.subplots(figsize=(10, 4.5), layout='constrained')
    for (run, result), color in zip(heat_records, colors, strict=True):
        frequencies = result['frequencies_cm1'].reshape(-1)
        frequencies = frequencies[np.isfinite(frequencies) & (np.abs(frequencies) > ZERO_FREQUENCY_TOLERANCE_CM1)]
        ax.hist(frequencies, bins=80, density=True, histtype='step', color=color, label=f"{run['temperature_K']:g} K MD")
    ax.set(xlabel=r'Frequency [cm$^{-1}$]', ylabel='Probability density', title='PET-SOL nonzero vibrational frequencies')
    ax.legend()
    plt.show()

## Sample one PET-SOL trajectory

Change `SELECTED_TEMPERATURE_K` and `TRAJECTORY_STRIDE` in the settings cell when needed. The optional Chemiscope viewer never reads a PET-MAD trajectory.

In [ ]:
trajectory_runs = [run for run in runs if run['trajectories']]
matches = [run for run in trajectory_runs if run['temperature_K'] == SELECTED_TEMPERATURE_K]
if not matches:
    print(f'No PET-SOL trajectory found at {SELECTED_TEMPERATURE_K} K.')
else:
    selected_run = matches[0]
    trajectory_path = selected_run['trajectories'][0]
    import ase.io

    viewer_frames = ase.io.read(trajectory_path, index=f'::{TRAJECTORY_STRIDE}')
    print(f'Selected: {trajectory_path.relative_to(PROJECT_DIR)}')
    print(f'Sampled {len(viewer_frames)} frames with {len(viewer_frames[0])} atoms each')
    force_frames = []
    maximum_forces = []
    for frame in viewer_frames:
        try:
            forces = frame.get_forces()
        except Exception:
            force_frames = []
            maximum_forces = []
            break
        force_frames.append(forces)
        maximum_forces.append(np.linalg.norm(forces, axis=1).max())
    if maximum_forces:
        plt.figure(figsize=(10, 3.5), layout='constrained')
        plt.plot(np.arange(len(maximum_forces)) * TRAJECTORY_STRIDE, maximum_forces, marker='o', markersize=3)
        plt.xlabel('Trajectory frame')
        plt.ylabel('Maximum force [eV/Å]')
        plt.title(f"PET-SOL maximum force — {SELECTED_TEMPERATURE_K:g} K")
        plt.show()
    else:
        print('Forces are not stored in the trajectory.')

    try:
        import chemiscope
    except ImportError:
        print(f'Chemiscope is not installed in this kernel: {sys.executable}')
        print('Select the mof-heat-capacity-izar kernel or install chemiscope in the active kernel.')
    else:
        frame_count = len(viewer_frames)
        properties = {'sampled frame': np.arange(frame_count)}
        selected_md = selected_run['md'][0] if selected_run['md'] else None
        if selected_md is not None:
            sample_indices = np.minimum(
                np.arange(frame_count) * TRAJECTORY_STRIDE, len(selected_md['time_ps']) - 1
            )
            properties.update({
                'time': {'target': 'structure', 'values': selected_md['time_ps'][sample_indices], 'units': 'ps'},
                'potential energy': {'target': 'structure', 'values': selected_md['potential_energy_eV'][sample_indices], 'units': 'eV'},
                'temperature': {'target': 'structure', 'values': selected_md['temperature_K'][sample_indices], 'units': 'K'},
            })
        shapes = {}
        if force_frames and len(force_frames) == frame_count:
            for frame, forces in zip(viewer_frames, force_frames, strict=True):
                frame.arrays['forces_eV_per_A'] = forces
            shapes['forces'] = chemiscope.ase_vectors_to_arrows(
                viewer_frames, 'forces_eV_per_A', scale=1.0, radius=0.08
            )
        structure_settings = {'keepOrientation': True, 'playbackDelay': 150}
        if shapes:
            structure_settings['shape'] = 'forces'
        viewer = chemiscope.show(
            structures=viewer_frames,
            properties=properties,
            shapes=shapes,
            settings={'structure': [structure_settings], 'map': {'joinPoints': True}},
            mode='default',
        )
        display(viewer)
        export_path = selected_run['directory'] / f'{trajectory_path.stem}-sampled.json.gz'
        viewer.save(str(export_path))
        print(f'Saved: {export_path.relative_to(PROJECT_DIR)}')

## Interpretation checklist

Before treating the results as converged properties, verify temperature stability, reasonable structures and forces, heat-capacity stability across decorrelated frames, vibrational zero/imaginary modes, and convergence with respect to MD duration and Hessian settings. These short calculations primarily validate the PET-SOL workflow.